# **260202/ml/03_지도학습(분류)**

# **3-1. 들어가며**

## **들어가며**

안녕하세요 여러분. 이번 시간에는 지도학습(분류) 모델에 대해 알아보도록 하겠습니다.

지도학습(분류) 모델들을 살펴보고, 교차검증 및 분류 모델을 평가하는 방법에 대해 알아보겠습니다.

## **학습목표**

지도학습(분류) 모델을 활용할 수 있다.하이퍼파라미터 튜닝을 할 수 있다.모델을 평가할 수 있다.

- 지도학습(분류) 모델을 활용할 수 있다.
- 하이퍼파라미터 튜닝을 할 수 있다.
- 모델을 평가할 수 있다.

## **오늘의 주제**

지도학습(분류)이 무엇인지, 어떻게 모델을 만들고 하이퍼파라미터를 튜닝하는지, 그리고 모델 평가는 어떻게 하는지 살펴보겠습니다.

# **3-2. 의사결정나무**

## **이번 시간 정리**

- **지도학습 알고리즘 (분류, 회귀)**
- **직관적인 알고리즘 (이해 쉬움)**
- **과대적합되기 쉬운 알고리즘 (트리 깊이 제한 필요)**

In [ ]:
# 라이브러리 불러오기
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
# 데이터 생성
from sklearn.datasets import load_breast_cancer

def make_dataset():
    cancer = load_breast_cancer()
    df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
    df['target'] = cancer.target
    X_train, X_test, y_train, y_test = train_test_split(
        df.drop('target', axis=1), df['target'], test_size=0.5, random_state=1004)
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = make_dataset()
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((284, 30), (285, 30), (284,), (285,))

In [ ]:
# 타겟 확인
y_train.value_counts()

,count
target,
1,190
0,94


## 의사결정나무

- 지도학습(분류)에서 가장 유용하게 사용되고 있는 기법 중 하나입니다.
- 트리의 루트(root)에서 시작해서 정보이득이 최대가 되는 특성으로 데이터를 나눕니다.
- 정보이득(information gain)이 최대가 되는 특성을 나누는 기준(불순도를 측정하는 기준)은 '지니'와 '엔트로피'가 사용됩니다.
  - 데이터가 한 종류만 있다면 엔트로피/지니 불순도는 0에 가깝고, 서로 다른 데이터의 비율이 비슷하면 1에 가깝습니다.
  - 정보이득(information gain)이 최대라는 것은 불순도를 최소화 하는 방향입니다. (1-불순도)

In [ ]:
# 의사결정나무
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(random_state=0)
model.fit(X_train, y_train)

pred = model.predict(X_test)

accuracy_score(y_test, pred)

0.9263157894736842

## 의사결정나무 하이퍼파라미터

In [ ]:
# 의사결정나무 하이퍼 파라미터

"""
- criterion (기본값 gini) : 불순도 지표 (또는 엔트로피 불순도 entropy)
- max_depth (기본값 None) : 최대 한도 깊이
- min_samples_split (기본값 2) : 자식 노드를 갖기 위한 최소한의 데이터 수
- min_samples_leaf (기본값 1) : 리프 노드가 되기 위한 최소 샘플 수
"""

from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(criterion= 'entropy',
                               max_depth = 6,
                               min_samples_split=2,
                               min_samples_leaf=2,
                               random_state=0)
model.fit(X_train, y_train)

pred = model.predict(X_test)

accuracy_score(y_test, pred)


0.9228070175438596

# **3-3. 랜덤포레스트**

## **이번 시간 정리**

- 앙상블 방법
  - 배깅: 같은 알고리즘으로 여러 모델을 만들어 분류함(랜덤포레스트)
  - 부스팅: 학습과 예측을 하면서 가중치 반영 (xgboost)

## **랜덤포레스트**

In [ ]:
# 랜덤포레스트
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state=0)
model.fit(X_train, y_train)
pred = model.predict(X_test)
accuracy_score(y_test, pred)

0.9438596491228071

In [ ]:
# 랜덤포레스트 하이퍼파라미터
"""
n_estimators (기본값 100) : 트리의 수
criterion (기본값 gini) : 불순도 지표
max_depth (기본값 None) : 최대 한도 깊이
min_samples_split (기본값 2) : 자식 노드를 갖기 위한 최소한의 데이터 수
min_samples_leaf (기본값 1) : 리프 노드가 되기 위한 최소 샘플 수
"""
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=500,
                               max_depth = 5,
                               random_state=0)
model.fit(X_train, y_train)
pred = model.predict(X_test)
accuracy_score(y_test, pred)

# max_depth 는 변경되어도 큰 차이 없음


0.9473684210526315

# **3-4. xgboost**

## 이번 시간 정리

- 트리 앙상블 중 성능이 좋은 알고리즘
- eXtreme Gradient Boosting를 줄여서 XGBoost라고 한다.
- 약한 학습기가 계속해서 업데이트를 하며 좋은 모델을 만들어 간다.
- 부스팅(앙상블) 기반의 알고리즘
- 캐글(글로벌 AI 경진대회)에서 뛰어난 성능을 보이면서 인기가 높아짐

## XGBoost

In [ ]:
# xgboost
from xgboost import XGBClassifier
model = XGBClassifier(random_state=0,
                      use_label_encoder=False,
                      eval_metric='logloss')
model.fit(X_train, y_train)
pred = model.predict(X_test)
accuracy_score(y_test, pred)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [02:33:29] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0.9614035087719298

In [ ]:
# xgboost 하이퍼 파라미터
"""
booster (기본값 gbtree) : 부스팅 알고리즘 (또는 dart, gblinear)
objective (기본값 binary:logistic) : 이진분류 (다중분류: multi:softmax)
max_depth (기본값 6) : 최대 한도 깊이
-------------------------------------------
* learning_rate (기본값 0.1) : 학습률
* n_estimators (기본값 100) : 트리의 수
-------------------------------------------
subsample (기본값 1) : 훈련 샘플 개수의 비율
colsample_bytree (기본값 1) : 특성 개수의 비율
n_jobs (기본값 1) : 사용 코어 수 (-1: 모든 코어를 다 사용)
"""
from xgboost import XGBClassifier
model = XGBClassifier(random_state=0,
                      use_label_encoder=False,
                      eval_metric ='logloss',
                      bosster = 'gdtree',
                      objective = 'binary:logistic',
                      max_depth =5,
                      learning_rate = 0.05,
                      n_estimators = 500,
                      subsample = 1,
                      colsample_bytree = 1,
                      n_jobs = -1
                      )
model.fit(X_train, y_train)
pred = model.predict(X_test)
accuracy_score(y_test, pred)

# default :0.9614035087719298

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [02:44:59] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "bosster", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0.9614035087719298

In [ ]:
# 조기종료
from xgboost import XGBClassifier
model = XGBClassifier(random_state=0,
                      use_label_encoder=False,
                      eval_metric ='logloss',
                      learning_rate = 0.05,
                      n_estimators = 500,
                      early_stopping_rounds=10 # 교제에서는 model.fit() 함수에 들어가지만 버전문제로 에러 발생
                      )
eval_set = [(X_test, y_test)]
model.fit(X_train, y_train, eval_set=eval_set)
pred = model.predict(X_test)
accuracy_score(y_test, pred)

[0]	validation_0-logloss:0.65309
[1]	validation_0-logloss:0.61762
[2]	validation_0-logloss:0.58594
[3]	validation_0-logloss:0.55672
[4]	validation_0-logloss:0.53141
[5]	validation_0-logloss:0.50589
[6]	validation_0-logloss:0.48413
[7]	validation_0-logloss:0.46301
[8]	validation_0-logloss:0.44362
[9]	validation_0-logloss:0.42760
[10]	validation_0-logloss:0.41150
[11]	validation_0-logloss:0.39625
[12]	validation_0-logloss:0.38137
[13]	validation_0-logloss:0.36927
[14]	validation_0-logloss:0.35623
[15]	validation_0-logloss:0.34596
[16]	validation_0-logloss:0.33446
[17]	validation_0-logloss:0.32360
[18]	validation_0-logloss:0.31385
[19]	validation_0-logloss:0.30428
[20]	validation_0-logloss:0.29609
[21]	validation_0-logloss:0.28866
[22]	validation_0-logloss:0.28066
[23]	validation_0-logloss:0.27380
[24]	validation_0-logloss:0.26678
[25]	validation_0-logloss:0.26129
[26]	validation_0-logloss:0.25525
[27]	validation_0-logloss:0.24966
[28]	validation_0-logloss:0.24555
[29]	validation_0-loglos

/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:386: UserWarning: [02:52:25] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[49]	validation_0-logloss:0.18711
[50]	validation_0-logloss:0.18575
[51]	validation_0-logloss:0.18375
[52]	validation_0-logloss:0.18250
[53]	validation_0-logloss:0.18138
[54]	validation_0-logloss:0.17952
[55]	validation_0-logloss:0.17851
[56]	validation_0-logloss:0.17783
[57]	validation_0-logloss:0.17625
[58]	validation_0-logloss:0.17602
[59]	validation_0-logloss:0.17554
[60]	validation_0-logloss:0.17379
[61]	validation_0-logloss:0.17313
[62]	validation_0-logloss:0.17266
[63]	validation_0-logloss:0.17257
[64]	validation_0-logloss:0.17210
[65]	validation_0-logloss:0.17096
[66]	validation_0-logloss:0.17088
[67]	validation_0-logloss:0.17095
[68]	validation_0-logloss:0.17029
[69]	validation_0-logloss:0.16975
[70]	validation_0-logloss:0.16997
[71]	validation_0-logloss:0.16952
[72]	validation_0-logloss:0.16931
[73]	validation_0-logloss:0.16805
[74]	validation_0-logloss:0.16806
[75]	validation_0-logloss:0.16814
[76]	validation_0-logloss:0.16778
[77]	validation_0-logloss:0.16767
[78]	validatio

0.9508771929824561

# **3-5. 교차검증**

- 일반적으로 모델을 학습시킬 때 데이터를 train set과 test set으로 나누어 train set을 가지고 학습을 수행합니다.
- 교차검증이란 여기서 train set을 다시 train set과 validation set으로 나누어 학습 중 검증과 수정을 수행하는 것을 의미합니다.

## **Kfold**

In [ ]:
# kfold
from sklearn.model_selection import KFold
model = DecisionTreeClassifier(random_state=0)

kfold= KFold(n_splits=5)
for train_idx, test_idx in kfold.split(X):
  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

  model.fit(X_train, y_train)
  pred = model.predict(X_test)
  print(accuracy_score(y_test, pred))

0.8771929824561403
0.9122807017543859
0.9473684210526315
0.9385964912280702
0.8407079646017699


## **StratifiedKfold**

In [ ]:
# StratifiedKfold
# 불균형한 타겟 비율을 가진 데이터가 한쪽으로 치우치는 것을 방지

from sklearn.model_selection import StratifiedKFold
model = DecisionTreeClassifier(random_state=0)

kfold= StratifiedKFold(n_splits=5)
for train_idx, test_idx in kfold.split(X,y):
  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

  model.fit(X_train, y_train)
  pred = model.predict(X_test)
  print(accuracy_score(y_test, pred))



0.9035087719298246
0.9210526315789473
0.9122807017543859
0.9473684210526315
0.9026548672566371


## 사이킷런 교차검증

In [ ]:
# 교차검증
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=3)
scores

array([0.88947368, 0.94210526, 0.86243386])

In [ ]:
# 평균점수
scores.mean()

np.float64(0.8980042699340944)

In [ ]:
# 교차검증 StratifiedKfold
kfold = StratifiedKFold(n_splits=5)
scores = cross_val_score(model, X, y, cv=kfold)
scores

array([0.90350877, 0.92105263, 0.9122807 , 0.94736842, 0.90265487])

In [ ]:
# 평균점수
scores.mean()

np.float64(0.9173730787144851)

# **3-6. 평가(분류)**

## **평가 (분류 모델)**

- 정확도 accuracy: 실제 값과 예측값이 일치하는 비율
- 정밀도 precision: 양성이라고 예측한 값 중 실제 양성인 값의 비율 (암이라고 예측 한 값 중 실제 암)
- 재현율 recall: 실제 양성 값 중 양성으로 예측한 값의 비율 (암을 암이라고 판단)

---
- F1: 정밀도와 재현율의 조화평균
---
- ROC-AUC
  - ROC: 참 양성 비율(True Positive Rate)에 대한 거짓 양성 비율(False Positive Rate) 곡선
  - AUC: ROC곡선 면적 아래 (완벽하게 분류되면 AUC가 1임)

In [ ]:
# 정확도
from sklearn.metrics import accuracy_score
accuracy_score(y_test, pred)

0.9026548672566371

In [ ]:
# 재현율
from sklearn.metrics import recall_score
recall_score(y_test, pred)

0.8873239436619719

In [ ]:
# f1 score
from sklearn.metrics import f1_score
f1_score(y_test, pred)

0.9197080291970803

In [ ]:
# roc_auc
from sklearn.metrics import roc_auc_score
# xgboost
from xgboost import XGBClassifier
model = XGBClassifier(random_state=0,
                      use_label_encoder=False,
                      eval_metric='logloss')
model.fit(X_train, y_train)
pred = model.predict_proba(X_test) # 예측결과를 백분율로
pred

roc_auc_score(y_test, pred[: ,1]) # [: ,1] -> 전체 데이터, 1

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [03:20:08] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


np.float64(0.9976525821596244)